# S4 Portfolio — Supply-Chain Hotspot Analysis & Radar Profiles

This notebook answers: **"Which projects are outliers, and what makes them risky?"**

Every stressor is **Z-score normalised against the portfolio average** so projects are compared
on equal footing regardless of investment size.

## Three analytical layers per project

| Layer | Description | Signal |
|-------|-------------|--------|
| **Baseline** | Raw Leontief supply-chain (Tier 0–10) | Absolute burden |
| **Scenario-Adjusted** | SSP2-4.5 decarbonisation trajectory (2030) | Decoupling from grid |
| **Dependency-Weighted** | ENCORE × WWF × SC sensitivity applied | Ecosystem amplification |

**Hotspot rule:** if the dependency multiplier on any stressor exceeds the `DEP_THRESHOLD`,
the cell / radar fill turns **red** — flagging local ecosystem vulnerability.

In [1]:
# ══════════════════════════════════════════════════════════════════
# PARAMETERS
# ══════════════════════════════════════════════════════════════════
from pathlib import Path

RESULTS_DIR    = Path("results")
SCENARIO       = "SSP2-4.5"        # scenario layer for adjusted / dep-weighted
FOCUS_YEAR     = 2030              # analysis year
MODEL          = "OSeMOSYS"
DEP_THRESHOLD  = 1.30              # dep multiplier above which → hotspot (red flag)
ZSCORE_CLIP    = 2.5               # clip Z-scores for colour scale readability

In [2]:
# ══════════════════════════════════════════════════════════════════
# IMPORTS
# ══════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

# Stressor config: (col_suffix, display_name, unit, polarity, radar_axis_label)
STRESSORS = [
    ("GHG",        "GHG",        "tCO₂e",  "neg", "GHG [−]"),
    ("Employment", "Employment", "FTE",     "pos", "Jobs [+]"),
    ("Water",      "Water",      "000 m³",  "neg", "Water [−]"),
    ("VA",         "Value Added","M USD",   "pos", "VA [+]"),
]

# Map stressor key → exact column suffix in each CSV
COL_BASE = {   # supply_chain_summary
    "GHG":        "GHG_tCO2e_t0_10",
    "Employment": "Emp_FTE_t0_10",
    "Water":      "Water_1000m3_t0_10",
    "VA":         "VA_Musd_t0_10",
}
COL_ADJ = {   # scenario_weighted_summary   (_adj suffix)
    "GHG":        "GHG_adj_tCO2e",
    "Employment": "Employment_adj_FTE",
    "Water":      "Water_adj_1000m3",
    "VA":         "ValueAdded_adj_M$",
}
COL_DEP = {   # dep_weighted_summary
    "GHG":        "GHG_dep_tCO2e",
    "Employment": "Employment_dep_FTE",
    "Water":      "Water_dep_1000m3",
    "VA":         "ValueAdded_dep_M$",
}

NEG_COLOR  = "#d62728"
POS_COLOR  = "#2ca02c"
BLUE       = "#4e79a7"
ORANGE     = "#f28e2b"
print("Imports OK")

Imports OK


In [3]:
# ══════════════════════════════════════════════════════════════════
# LOAD DATA
# ══════════════════════════════════════════════════════════════════
base_df = pd.read_csv(RESULTS_DIR / "supply_chain_summary.csv")
sw_df   = pd.read_csv(RESULTS_DIR / "scenario_weighted_summary.csv")
dw_df   = pd.read_csv(RESULTS_DIR / "dep_weighted_summary.csv")

# Filter to chosen scenario / year / model
sw = sw_df[
    (sw_df["scenario"] == SCENARIO) &
    (sw_df["year"]     == float(FOCUS_YEAR)) &
    (sw_df["model"]    == MODEL)
].copy()
dw = dw_df[
    (dw_df["scenario"] == SCENARIO) &
    (dw_df["year"]     == float(FOCUS_YEAR)) &
    (dw_df["model"]    == MODEL)
].copy()

PROJECTS = sorted(base_df["project_id"].unique())
print(f"Projects: {PROJECTS}")
print(f"Baseline rows: {len(base_df)} | Scenario rows: {len(sw)} | Dep rows: {len(dw)}")

Projects: ['Hydro_AF', 'Hydro_AS', 'Hydro_EU', 'Proj_001', 'Proj_002', 'Proj_003', 'Rail_EU_DEV', 'Rail_EU_OP1', 'Rail_EU_OP2']
Baseline rows: 9 | Scenario rows: 9 | Dep rows: 9


In [4]:
# ══════════════════════════════════════════════════════════════════
# BUILD MASTER ANALYSIS TABLE
# Columns: project_id, asset_class, region,
#          {s}_base, {s}_adj, {s}_dep, dep_eff_{s}   for each stressor
# ══════════════════════════════════════════════════════════════════

master = base_df[["project_id", "asset_class", "region"]].copy()

# Baseline
for s, col in COL_BASE.items():
    master[f"{s}_base"] = base_df[col].values

# Scenario-adjusted
sw_sub = sw[["project_id"] + list(COL_ADJ.values())].copy()
sw_sub = sw_sub.rename(columns={v: f"{k}_adj" for k, v in COL_ADJ.items()})
master = master.merge(sw_sub, on="project_id", how="left")

# Dependency-weighted
dw_sub = dw[["project_id"] + list(COL_DEP.values())].copy()
dw_sub = dw_sub.rename(columns={v: f"{k}_dep" for k, v in COL_DEP.items()})
master = master.merge(dw_sub, on="project_id", how="left")

# Effective dependency multiplier: dep / adj  (1.0 = neutral)
for s in ["GHG", "Employment", "Water", "VA"]:
    master[f"dep_eff_{s}"] = (master[f"{s}_dep"] / master[f"{s}_adj"]).round(3)

# Hotspot flag: ANY stressor dep_eff > threshold
master["is_hotspot"] = master[[f"dep_eff_{s}" for s in ["GHG","Employment","Water","VA"]]].gt(DEP_THRESHOLD).any(axis=1)

master = master.set_index("project_id")
print(master[["asset_class","region"] + [f"dep_eff_{s}" for s in ["GHG","Employment","Water","VA"]] + ["is_hotspot"]].to_string())

            asset_class  region  dep_eff_GHG  dep_eff_Employment  dep_eff_Water  dep_eff_VA  is_hotspot
project_id                                                                                             
Hydro_AF         Energy  Africa        1.032               1.112          1.510       1.286        True
Hydro_AS         Energy    Asia        1.046               1.128          1.557       1.329        True
Hydro_EU         Energy  Europe        1.016               1.109          1.407       1.263        True
Proj_001         Health   LATAM        1.003               1.027          1.352       1.218        True
Proj_002         Health  Africa        0.995               1.009          1.392       1.234        True
Proj_003         Health  Europe        0.985               1.029          1.290       1.214       False
Rail_EU_DEV   Transport  Europe        0.934               1.164          1.218       1.125       False
Rail_EU_OP1   Transport  Europe        0.966               1.143

In [5]:
# ══════════════════════════════════════════════════════════════════
# Z-SCORE NORMALISATION
# Each stressor × layer normalised against portfolio mean / std
# ══════════════════════════════════════════════════════════════════
def zscore_col(series: pd.Series) -> pd.Series:
    """Z-score, safe against std=0."""
    std = series.std(ddof=0)
    return ((series - series.mean()) / std).clip(-ZSCORE_CLIP, ZSCORE_CLIP) if std > 0 else series * 0

z = pd.DataFrame(index=master.index)
layers = [("base", "Baseline"), ("adj", f"{SCENARIO} {FOCUS_YEAR}"), ("dep", "Dep-Weighted")]
for s, sname, sunit, polarity, _ in STRESSORS:
    for suffix, lname in layers:
        col = f"{s}_{suffix}"
        z[f"{s}_{suffix}_z"] = zscore_col(master[col])

print("Z-score table (first 4 cols):")
print(z.iloc[:, :4].round(2).to_string())

Z-score table (first 4 cols):
             GHG_base_z  GHG_adj_z  GHG_dep_z  Employment_base_z
project_id                                                      
Hydro_AF          -0.41      -0.43      -0.43              -0.43
Hydro_AS          -0.10      -0.16      -0.14              -0.09
Hydro_EU          -0.50      -0.49      -0.50              -0.52
Proj_001           0.04       0.12       0.15               0.14
Proj_002          -0.43      -0.44      -0.45              -0.44
Proj_003          -0.39      -0.38      -0.39              -0.39
Rail_EU_DEV        2.50       2.50       2.50               2.50
Rail_EU_OP1       -0.50      -0.49      -0.50              -0.52
Rail_EU_OP2       -0.50      -0.49      -0.50              -0.52


In [6]:
# ══════════════════════════════════════════════════════════════════
# SECTION 1 — PORTFOLIO HOTSPOT HEATMAP  (Z-score grid)
# Rows: projects  |  Columns: stressor × layer
# Colour: red = above average burden, blue = below average
# ✦ markers where dep_eff > DEP_THRESHOLD (local ecosystem flag)
# ══════════════════════════════════════════════════════════════════

# Build ordered column list and labels
cols, col_labels = [], []
for s, sname, _, polarity, _ in STRESSORS:
    for suffix, lname in layers:
        cols.append(f"{s}_{suffix}_z")
        sign = "[−]" if polarity == "neg" else "[+]"
        col_labels.append(f"{sign}{sname}<br>{lname}")

# Matrix
Z_mat = z[cols].values

# Hotspot annotation overlay
annot_text = [["" for _ in cols] for _ in PROJECTS]
annot_color = [["black" for _ in cols] for _ in PROJECTS]
for pi, pid in enumerate(PROJECTS):
    col_idx = 0
    for si, (s, *_) in enumerate(STRESSORS):
        eff = master.loc[pid, f"dep_eff_{s}"]
        for li, (suffix, _) in enumerate(layers):
            val = z.loc[pid, f"{s}_{suffix}_z"]
            annot_text[pi][col_idx]  = f"{val:.1f}{'✦' if (eff > DEP_THRESHOLD and suffix=='dep') else ''}"
            annot_color[pi][col_idx] = NEG_COLOR if (eff > DEP_THRESHOLD and suffix == "dep") else "white" if abs(val) > 1 else "#333"
            col_idx += 1

fig1 = go.Figure(go.Heatmap(
    z=Z_mat,
    x=col_labels,
    y=PROJECTS,
    colorscale=[
        [0.0,  "#4575b4"],   # strongly below average (low burden)
        [0.35, "#91bfdb"],
        [0.5,  "#ffffbf"],   # at portfolio average
        [0.65, "#fc8d59"],
        [1.0,  "#d73027"],   # strongly above average (high burden)
    ],
    zmin=-ZSCORE_CLIP, zmax=ZSCORE_CLIP,
    text=annot_text,
    texttemplate="%{text}",
    textfont=dict(size=9),
    colorbar=dict(title="Z-score", tickvals=[-2, -1, 0, 1, 2]),
))

# Override text colours cell by cell via separate scatter overlay
for pi, pid in enumerate(PROJECTS):
    for ci, (suffix, _) in enumerate([("dep", "")]*0):  # placeholder, real annotations below
        pass

# Add ✦ markers for dep hotspots as separate scatter
hx, hy = [], []
for pi, pid in enumerate(PROJECTS):
    col_offset = 0
    for si, (s, *_) in enumerate(STRESSORS):
        eff = master.loc[pid, f"dep_eff_{s}"]
        dep_col_idx = col_offset + 2  # index of the 'dep' layer column
        if eff > DEP_THRESHOLD:
            hx.append(col_labels[dep_col_idx])
            hy.append(pid)
        col_offset += 3

if hx:
    fig1.add_trace(go.Scatter(
        x=hx, y=hy,
        mode="markers",
        marker=dict(symbol="star", size=14, color=NEG_COLOR, line=dict(color="white", width=1)),
        name=f"Hotspot (dep > {DEP_THRESHOLD}×)",
        showlegend=True,
    ))

fig1.update_layout(
    title=dict(
        text=(
            f"<b>Portfolio Hotspot Heatmap — Z-score normalised</b>"
            f"<br><span style='font-size:11px;color:#555'>"
            f"Scenario: {SCENARIO} | Year: {FOCUS_YEAR} | "
            f"★ = dep multiplier &gt; {DEP_THRESHOLD}× (ecosystem vulnerability flag)</span>"
        ),
        font=dict(size=14),
    ),
    height=420,
    xaxis=dict(tickangle=-35, tickfont=dict(size=9)),
    yaxis=dict(tickfont=dict(size=10)),
    margin=dict(l=120, r=60, t=100, b=120),
    paper_bgcolor="#fafafa",
    font=dict(family="Arial", size=10),
    legend=dict(orientation="h", y=-0.35),
)
fig1.show()

In [7]:
# ══════════════════════════════════════════════════════════════════
# SECTION 2 — DEPENDENCY AMPLIFICATION BARS
# Shows effective dep multiplier per project × stressor
# Red fill where dep_eff > threshold; dotted line at 1.0 (neutral)
# ══════════════════════════════════════════════════════════════════
stressor_keys = [s for s, *_ in STRESSORS]
stressor_names = {s: sname for s, sname, *_ in STRESSORS}
stressor_pol   = {s: p for s, _, __, p, ___ in STRESSORS}

fig2 = make_subplots(
    rows=1, cols=len(stressor_keys),
    subplot_titles=[f"dep_eff — {stressor_names[s]}" for s in stressor_keys],
    shared_yaxes=True,
)

proj_order = list(master.sort_values("dep_eff_Water", ascending=True).index)

for ci, s in enumerate(stressor_keys, start=1):
    col_key = f"dep_eff_{s}"
    vals    = master.loc[proj_order, col_key]
    colors  = [NEG_COLOR if v > DEP_THRESHOLD else "#aec7e8" for v in vals]

    fig2.add_trace(go.Bar(
        x=vals.values,
        y=proj_order,
        orientation="h",
        marker_color=colors,
        marker_line=dict(color="white", width=0.5),
        text=[f"{v:.2f}×" for v in vals],
        textposition="outside",
        textfont=dict(size=9),
        name=stressor_names[s],
        showlegend=False,
    ), row=1, col=ci)

    # Neutral line at 1.0
    fig2.add_vline(x=1.0, line_dash="dot", line_color="#555", line_width=1, row=1, col=ci)
    # Threshold line
    fig2.add_vline(x=DEP_THRESHOLD, line_dash="dash", line_color=NEG_COLOR, line_width=1.5, row=1, col=ci)

fig2.update_layout(
    title=(
        f"<b>Dependency Amplification</b> — dep-weighted ÷ scenario-adjusted "
        f"({SCENARIO} {FOCUS_YEAR})"
        f"<br><span style='font-size:11px;color:#555'>"
        f"Red = dep multiplier exceeds {DEP_THRESHOLD}× threshold | dashed = threshold | dotted = neutral 1.0</span>"
    ),
    height=380,
    margin=dict(l=110, r=80, t=100, b=40),
    paper_bgcolor="#fafafa",
    font=dict(family="Arial", size=10),
    bargap=0.3,
)
fig2.show()

In [8]:
# ══════════════════════════════════════════════════════════════════
# SECTION 3 — RADAR CHART BUILDER
# Three layers per project: Baseline / Scenario-Adj / Dep-Weighted
# Values: percentile rank within portfolio (0–1)
#   Negative stressors (GHG, Water): high rank = high burden (bad → red)
#   Positive stressors (Employment, VA): high rank = high benefit (good → green)
# Flash red outer fill if ANY stressor dep_eff > DEP_THRESHOLD
# ══════════════════════════════════════════════════════════════════

# Compute percentile ranks per stressor × layer across portfolio
pct = pd.DataFrame(index=master.index)
for s, sname, sunit, polarity, _ in STRESSORS:
    for suffix, lname in layers:
        col = f"{s}_{suffix}"
        rank = master[col].rank(pct=True, method="average")
        pct[f"{s}_{suffix}_pct"] = rank

# Radar axis labels and order
radar_axes = [ax for _, _, _, _, ax in STRESSORS]
# Close the polygon: repeat first axis at end
radar_axes_closed = radar_axes + [radar_axes[0]]


def build_radar(project_id: str) -> go.Figure:
    is_hot   = bool(master.loc[project_id, "is_hotspot"])
    asset    = master.loc[project_id, "asset_class"]
    region   = master.loc[project_id, "region"]

    layer_cfg = [
        ("base", "Baseline",                "#7f7f7f", "dash",    0.10),
        ("adj",  f"{SCENARIO} {FOCUS_YEAR}", BLUE,      "dashdot", 0.15),
        ("dep",  "Dep-Weighted",
            NEG_COLOR if is_hot else ORANGE,
            "solid", 0.25 if is_hot else 0.15),
    ]

    fig = go.Figure()

    for suffix, lname, color, dash, alpha in layer_cfg:
        r_vals = [pct.loc[project_id, f"{s}_{suffix}_pct"] for s, *_ in STRESSORS]
        r_vals_closed = r_vals + [r_vals[0]]

        fig.add_trace(go.Scatterpolar(
            r=r_vals_closed,
            theta=radar_axes_closed,
            mode="lines+markers",
            name=lname,
            line=dict(color=color, width=2.5 if suffix == "dep" else 1.8, dash=dash),
            fill="toself",
            fillcolor=f"rgba({int(color[1:3],16)},{int(color[3:5],16)},{int(color[5:7],16)},{alpha})",
            marker=dict(size=7 if suffix == "dep" else 5, color=color),
        ))

    # Per-axis dep_eff annotation on hover
    for s, sname, _, polarity, axis_label in STRESSORS:
        eff = master.loc[project_id, f"dep_eff_{s}"]
        flag = " ✦ HOTSPOT" if eff > DEP_THRESHOLD else ""
        r_dep = pct.loc[project_id, f"{s}_dep_pct"]
        fig.add_trace(go.Scatterpolar(
            r=[r_dep],
            theta=[axis_label],
            mode="markers",
            marker=dict(
                size=14,
                symbol="star" if eff > DEP_THRESHOLD else "circle-open",
                color=NEG_COLOR if eff > DEP_THRESHOLD else ORANGE,
                line=dict(color="white", width=1),
            ),
            name=f"{sname} dep={eff:.2f}×{flag}",
            showlegend=False,
            hovertemplate=f"{sname}: dep_eff = {eff:.2f}×{flag}<extra></extra>",
        ))

    hotspot_note = (
        f"<span style='color:{NEG_COLOR};font-weight:bold'> ★ HOTSPOT — dep &gt; {DEP_THRESHOLD}×</span>"
        if is_hot else ""
    )
    fig.update_layout(
        title=dict(
            text=(
                f"<b>{project_id}</b> · {asset} · {region}"
                f"{hotspot_note}"
                f"<br><span style='font-size:10px;color:#555'>"
                f"Percentile rank within portfolio (0 = lowest, 1 = highest burden/benefit)"
                f"</span>"
            ),
            font=dict(size=13),
            x=0.01, xanchor="left",
        ),
        polar=dict(
            radialaxis=dict(
                visible=True, range=[0, 1],
                tickvals=[0.25, 0.5, 0.75],
                ticktext=["P25", "P50", "P75"],
                tickfont=dict(size=8),
                gridcolor="#ddd",
                linecolor="#ccc",
            ),
            angularaxis=dict(
                tickfont=dict(size=10, color="#333"),
                linecolor="#ccc",
                gridcolor="#eee",
            ),
            bgcolor="white" if not is_hot else "#fff5f5",
        ),
        height=500,
        margin=dict(l=60, r=60, t=110, b=60),
        paper_bgcolor="#fafafa",
        legend=dict(orientation="h", y=-0.12, font=dict(size=10)),
        font=dict(family="Arial", size=10),
    )
    return fig

print("Radar builder defined.")

Radar builder defined.


In [9]:
# ══════════════════════════════════════════════════════════════════
# SECTION 4 — RENDER ONE RADAR PER PROJECT
# Hotspot projects shown first (sorted by max dep_eff desc)
# ══════════════════════════════════════════════════════════════════
eff_cols  = [f"dep_eff_{s}" for s in stressor_keys]
proj_sorted = (
    master[eff_cols]
    .max(axis=1)
    .sort_values(ascending=False)
    .index.tolist()
)

for pid in proj_sorted:
    fig = build_radar(pid)
    fig.show()

In [10]:
# ══════════════════════════════════════════════════════════════════
# SECTION 5 — PROJECT-SPECIFIC Z-SCORE HEATMAP (per-project)
# For each project: how does it deviate from portfolio on each
# stressor × layer?  Red = above portfolio average, blue = below.
# ══════════════════════════════════════════════════════════════════
layer_labels = ["Baseline", f"{SCENARIO} {FOCUS_YEAR}", "Dep-Weighted"]

for pid in proj_sorted:
    rows_z, rows_ann = [], []
    is_hot = bool(master.loc[pid, "is_hotspot"])

    for s, sname, sunit, polarity, _ in STRESSORS:
        row_vals, row_ann = [], []
        for suffix, lname in layers:
            zv  = float(z.loc[pid, f"{s}_{suffix}_z"])
            eff = float(master.loc[pid, f"dep_eff_{s}"])
            flag = "★" if (suffix == "dep" and eff > DEP_THRESHOLD) else ""
            row_vals.append(zv)
            row_ann.append(f"{zv:+.2f}{flag}")
        rows_z.append(row_vals)
        rows_ann.append(row_ann)

    stressor_labels_disp = [f"{sname}" for _, sname, *_ in STRESSORS]

    fig_ph = go.Figure(go.Heatmap(
        z=rows_z,
        x=layer_labels,
        y=stressor_labels_disp,
        colorscale=[
            [0.0, "#4575b4"], [0.35, "#91bfdb"],
            [0.5, "#ffffbf"],
            [0.65, "#fc8d59"], [1.0, "#d73027"],
        ],
        zmin=-ZSCORE_CLIP, zmax=ZSCORE_CLIP,
        text=rows_ann,
        texttemplate="%{text}",
        textfont=dict(size=12),
        colorbar=dict(title="Z-score"),
        showscale=True,
    ))

    asset  = master.loc[pid, "asset_class"]
    region = master.loc[pid, "region"]
    hotspot_tag = " ★ HOTSPOT" if is_hot else ""

    fig_ph.update_layout(
        title=dict(
            text=(
                f"<b>{pid}</b> — {asset} · {region}{hotspot_tag}"
                f"<br><span style='font-size:10px;color:#555'>"
                f"Z-score vs portfolio | ★ = dep multiplier &gt; {DEP_THRESHOLD}×"
                f"</span>"
            ),
            font=dict(size=13),
            x=0.01, xanchor="left",
        ),
        height=280,
        margin=dict(l=100, r=60, t=90, b=40),
        paper_bgcolor="#fff5f5" if is_hot else "#fafafa",
        font=dict(family="Arial", size=11),
    )
    fig_ph.show()

In [11]:
# ══════════════════════════════════════════════════════════════════
# SECTION 6 — HOTSPOT SUMMARY TABLE
# Ranked by max dep_eff; colour-coded cells for quick triage
# ══════════════════════════════════════════════════════════════════
summary = master[["asset_class", "region", "is_hotspot"] + eff_cols].copy()
summary.columns = [
    "Asset", "Region", "Hotspot",
    "dep_GHG ×", "dep_Jobs ×", "dep_Water ×", "dep_VA ×"
]
summary["Max dep ×"] = summary[["dep_GHG ×","dep_Jobs ×","dep_Water ×","dep_VA ×"]].max(axis=1).round(3)
summary = summary.sort_values("Max dep ×", ascending=False)

# Plotly table with conditional cell colours
def cell_colors(col):
    return [
        NEG_COLOR if v > DEP_THRESHOLD else "#eafbea" if v < 1.0 else "#fff"
        for v in summary[col]
    ]

fig_tbl = go.Figure(go.Table(
    header=dict(
        values=["<b>Project</b>", "<b>Asset</b>", "<b>Region</b>",
                "<b>Hotspot</b>",
                "<b>dep GHG</b>", "<b>dep Jobs</b>",
                "<b>dep Water</b>", "<b>dep VA</b>",
                "<b>Max dep</b>"],
        fill_color="#2c3e50",
        font=dict(color="white", size=11),
        align="center",
    ),
    cells=dict(
        values=[
            [f"<b>{p}</b>" for p in summary.index],
            summary["Asset"],
            summary["Region"],
            ["★ YES" if h else "—" for h in summary["Hotspot"]],
            summary["dep_GHG ×"].round(3),
            summary["dep_Jobs ×"].round(3),
            summary["dep_Water ×"].round(3),
            summary["dep_VA ×"].round(3),
            summary["Max dep ×"].round(3),
        ],
        fill_color=[
            ["#f0f0f0"] * len(summary),
            ["#f0f0f0"] * len(summary),
            ["#f0f0f0"] * len(summary),
            [NEG_COLOR if h else "#eafbea" for h in summary["Hotspot"]],
            cell_colors("dep_GHG ×"),
            ["#eafbea" if v > 1.0 else "#f0f0f0" for v in summary["dep_Jobs ×"]],
            cell_colors("dep_Water ×"),
            ["#eafbea" if v > 1.0 else "#f0f0f0" for v in summary["dep_VA ×"]],
            cell_colors("Max dep ×"),
        ],
        font=dict(size=11),
        align="center",
        height=28,
    )
))

fig_tbl.update_layout(
    title=dict(
        text=(
            f"<b>Hotspot Triage Table</b> — ranked by maximum dependency multiplier"
            f"<br><span style='font-size:10px;color:#555'>"
            f"Red cell = dep &gt; {DEP_THRESHOLD}× | Green = dep &lt; 1.0 (reduces risk)"
            f"</span>"
        ),
        font=dict(size=14),
    ),
    height=420,
    margin=dict(l=20, r=20, t=90, b=20),
    paper_bgcolor="#fafafa",
    font=dict(family="Arial"),
)
fig_tbl.show()